# Retina Ca²⁺ Waveform Activity Classification with XGBoost

This notebook shows a practical machine-learning workflow for classifying calcium-imaging waveforms from retina as **active** vs **non-active** using **Gradient Boosting / XGBoost**.

Assumed input files:

```text
data/uninjured_retina.csv   # ~50,000 waveforms
data/injured_retina.csv     # ~20,000 waveforms
```

The workflow is intentionally practical rather than perfect. It is designed to help you implement the method, inspect the data, train a model, evaluate it, and estimate activity thresholds such as ΔF/F ≈ 0.25.


## 1. Install required packages

Run this in your terminal inside your `.venv` or conda environment:

```bash
pip install pandas numpy matplotlib scikit-learn xgboost joblib jupyterlab ipykernel
```


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)
from sklearn.inspection import permutation_importance
from sklearn.ensemble import HistGradientBoostingClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

import joblib

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

uninjured_path = DATA_DIR / "uninjured_retina.csv"
injured_path = DATA_DIR / "injured_retina.csv"

print("XGBoost available:", XGBOOST_AVAILABLE)
print("Data directory:", DATA_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

## 2. Load the two CSV files

This section loads the two groups and adds a `retina_group` column:

- `uninjured`
- `injured`

Each row is assumed to represent one waveform or one cell-level waveform summary.

The notebook can work with either precomputed waveform features or raw time-series columns such as `t0`, `t1`, `t2`, etc.


In [ ]:
def load_retina_csv(path: Path, group_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Put the file at {path} or update the path in the notebook."
        )
    df = pd.read_csv(path)
    df["retina_group"] = group_name
    return df

uninjured_df = load_retina_csv(uninjured_path, "uninjured")
injured_df = load_retina_csv(injured_path, "injured")

df = pd.concat([uninjured_df, injured_df], ignore_index=True)

print("Combined shape:", df.shape)
display(df.head())
display(df.dtypes.head(30))

## 3. Identify active/non-active labels

For supervised ML, we need a target label:

```text
active = 1
non-active = 0
```

The notebook first searches for common label names.

If no label exists, it creates a provisional label using a simple ΔF/F rule:

```text
active = peak ΔF/F >= 0.25
```

You can later replace this with manual labels or curated labels.


In [ ]:
print("Columns:")
for c in df.columns:
    print(c)

In [ ]:
possible_label_cols = [
    "active", "is_active", "activity", "label", "class", "target",
    "active_label", "manual_label"
]

label_col = None
for col in possible_label_cols:
    if col in df.columns:
        label_col = col
        break

print("Detected label column:", label_col)

possible_dff_cols = [
    "peak_dff", "max_dff", "delta_f_over_f", "dff", "df_f",
    "deltaF_F", "delta_f_f", "amplitude", "peak_amplitude"
]

dff_col = None
for col in possible_dff_cols:
    if col in df.columns:
        dff_col = col
        break

print("Detected ΔF/F / amplitude column:", dff_col)

if label_col is None:
    if dff_col is None:
        raise ValueError(
            "No label column was found and no ΔF/F/amplitude column was found. "
            "Add a label column, or rename your peak ΔF/F column to 'peak_dff' or 'amplitude'."
        )
    ACTIVITY_DFF_THRESHOLD = 0.25
    df["active"] = (df[dff_col] >= ACTIVITY_DFF_THRESHOLD).astype(int)
    label_col = "active"
    print(f"Created label column 'active' using {dff_col} >= {ACTIVITY_DFF_THRESHOLD}")
else:
    df[label_col] = df[label_col].replace({
        "active": 1, "Active": 1, "ACTIVE": 1,
        "inactive": 0, "Inactive": 0, "INACTIVE": 0,
        "non-active": 0, "non_active": 0, "Non-active": 0,
        True: 1, False: 0
    }).astype(int)

print(df[label_col].value_counts(dropna=False))
display(df[[label_col, "retina_group"]].head())

## 4. Basic exploratory plots

These plots show class balance, group size, and the ΔF/F/amplitude distribution if an amplitude column is detected.


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
df[label_col].value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_title("Active vs Non-active Count")
ax.set_xlabel("Label: 0 = non-active, 1 = active")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
df["retina_group"].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Waveform Count by Retina Group")
ax.set_xlabel("Group")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

if dff_col is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    for group_name, sub in df.groupby("retina_group"):
        sub[dff_col].dropna().plot(kind="hist", bins=80, alpha=0.5, ax=ax, label=group_name)
    ax.axvline(0.25, linestyle="--", label="Example threshold = 0.25")
    ax.set_title(f"Distribution of {dff_col}")
    ax.set_xlabel(dff_col)
    ax.set_ylabel("Count")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Optional feature engineering from raw waveform columns

If the CSV contains time-series columns such as `t0`, `t1`, `t2` or `frame_0`, `frame_1`, this section computes simple waveform features:

- peak
- minimum
- mean
- standard deviation
- AUC
- peak index
- rise proxy
- decay proxy
- SNR proxy


In [ ]:
def detect_trace_columns(dataframe: pd.DataFrame):
    trace_cols = []
    for c in dataframe.columns:
        c_lower = str(c).lower()
        if c_lower.startswith("t") and c_lower[1:].isdigit():
            trace_cols.append(c)
        elif c_lower.startswith("frame_") and c_lower.replace("frame_", "").isdigit():
            trace_cols.append(c)
        elif c_lower.startswith("time_") and c_lower.replace("time_", "").isdigit():
            trace_cols.append(c)
    return trace_cols

trace_cols = detect_trace_columns(df)
print(f"Detected {len(trace_cols)} possible raw trace columns.")
print(trace_cols[:10])

def add_trace_features(dataframe: pd.DataFrame, trace_columns):
    out = dataframe.copy()
    if len(trace_columns) == 0:
        return out

    Xtrace = out[trace_columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)

    out["trace_peak"] = np.nanmax(Xtrace, axis=1)
    out["trace_min"] = np.nanmin(Xtrace, axis=1)
    out["trace_mean"] = np.nanmean(Xtrace, axis=1)
    out["trace_std"] = np.nanstd(Xtrace, axis=1)
    out["trace_auc"] = np.trapz(np.nan_to_num(Xtrace, nan=0.0), axis=1)
    out["trace_peak_index"] = np.nanargmax(np.nan_to_num(Xtrace, nan=-np.inf), axis=1)

    first_vals = Xtrace[:, 0]
    last_vals = Xtrace[:, -1]
    peaks = out["trace_peak"].to_numpy()
    peak_idx = out["trace_peak_index"].to_numpy()

    out["trace_rise_proxy"] = (peaks - first_vals) / np.maximum(peak_idx, 1)
    out["trace_decay_proxy"] = (peaks - last_vals) / np.maximum(Xtrace.shape[1] - peak_idx, 1)
    out["trace_snr_proxy"] = out["trace_peak"] / (out["trace_std"] + 1e-8)

    return out

df = add_trace_features(df, trace_cols)
display(df.head())

## 6. Select model features

The notebook uses numeric columns as model features and excludes labels, group text, IDs, and filenames.

`retina_group` is encoded as `retina_group_injured`, so the combined model can learn whether injury status shifts activity patterns.


In [ ]:
df_model = df.copy()
df_model["retina_group_injured"] = (df_model["retina_group"] == "injured").astype(int)

exclude_cols = {label_col, "retina_group"}

common_id_cols = [
    "cell_id", "mouse_id", "animal_id", "recording_id", "roi_id",
    "waveform_id", "file", "filename", "source_file", "condition"
]

for c in common_id_cols:
    if c in df_model.columns:
        exclude_cols.add(c)

numeric_cols = df_model.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

print("Number of selected features:", len(feature_cols))
print(feature_cols)

X = df_model[feature_cols].copy()
y = df_model[label_col].astype(int).copy()

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts(normalize=True))

## 7. Train/test split

If `mouse_id`, `animal_id`, `recording_id`, or a similar column exists, the notebook keeps all waveforms from the same animal/recording together to reduce data leakage.

Otherwise, it uses a standard stratified train/test split.


In [ ]:
group_split_col = None
for candidate in ["mouse_id", "animal_id", "recording_id", "file", "filename", "source_file"]:
    if candidate in df_model.columns:
        group_split_col = candidate
        break

print("Group split column:", group_split_col)

if group_split_col is not None:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(X, y, groups=df_model[group_split_col]))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train class balance:")
print(y_train.value_counts(normalize=True))
print("Test class balance:")
print(y_test.value_counts(normalize=True))

## 8. Train Gradient Boosting / XGBoost

Preferred model: `XGBClassifier`.

If `xgboost` is not installed, the notebook falls back to scikit-learn's `HistGradientBoostingClassifier`.

XGBoost learns many small decision trees sequentially; each new tree improves on mistakes made by previous trees.


In [ ]:
if XGBOOST_AVAILABLE:
    model = XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
else:
    model = HistGradientBoostingClassifier(
        max_iter=400,
        learning_rate=0.03,
        max_leaf_nodes=31,
        random_state=RANDOM_STATE,
    )

model.fit(X_train, y_train)
print("Model trained:", type(model).__name__)

## 9. Evaluate model performance

The model outputs `P(active | waveform features)`. The default decision cutoff is 0.5.


In [ ]:
def predict_proba_safe(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    scores = model.decision_function(X)
    return 1 / (1 + np.exp(-scores))

y_prob = predict_proba_safe(model, X_test)
y_pred = (y_prob >= 0.5).astype(int)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_prob),
    "average_precision": average_precision_score(y_test, y_prob),
}

display(pd.DataFrame([metrics]))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["non-active", "active"]))

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, label=f"ROC-AUC = {metrics['roc_auc']:.3f}")
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate / Recall")
ax.set_title("ROC Curve")
ax.legend()
plt.tight_layout()
plt.show()

precision, recall, pr_thresholds = precision_recall_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(recall, precision, label=f"Avg Precision = {metrics['average_precision']:.3f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Choose a model probability threshold

Instead of using only ΔF/F > 0.25, the classifier gives a probability.

Examples:

```text
P(active) > 0.50  # balanced default
P(active) > 0.80  # stricter, fewer false positives
P(active) > 0.90  # very strict active calls
```


In [ ]:
threshold_rows = []

for threshold in np.linspace(0.05, 0.95, 19):
    pred = (y_prob >= threshold).astype(int)
    threshold_rows.append({
        "probability_threshold": threshold,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "predicted_active_fraction": pred.mean(),
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(threshold_df["probability_threshold"], threshold_df["precision"], label="Precision")
ax.plot(threshold_df["probability_threshold"], threshold_df["recall"], label="Recall")
ax.plot(threshold_df["probability_threshold"], threshold_df["f1"], label="F1")
ax.set_xlabel("Model probability threshold")
ax.set_ylabel("Score")
ax.set_title("Choosing an Activity Probability Threshold")
ax.legend()
plt.tight_layout()
plt.show()

## 11. Estimate a practical ΔF/F threshold

If an amplitude or ΔF/F column exists, this scans possible ΔF/F cutoffs and finds the one with the best F1 score.

This lets you compare a traditional threshold like 0.25 against the ML model.


In [ ]:
if dff_col is not None:
    test_df = df_model.loc[X_test.index].copy()
    test_df["y_true"] = y_test.values
    test_df["model_probability_active"] = y_prob

    candidate_thresholds = np.linspace(
        np.nanpercentile(test_df[dff_col], 1),
        np.nanpercentile(test_df[dff_col], 99),
        100
    )

    dff_rows = []
    for thr in candidate_thresholds:
        pred = (test_df[dff_col] >= thr).astype(int)
        dff_rows.append({
            "dff_threshold": thr,
            "accuracy": accuracy_score(test_df["y_true"], pred),
            "precision": precision_score(test_df["y_true"], pred, zero_division=0),
            "recall": recall_score(test_df["y_true"], pred, zero_division=0),
            "f1": f1_score(test_df["y_true"], pred, zero_division=0),
            "predicted_active_fraction": pred.mean(),
        })

    dff_threshold_df = pd.DataFrame(dff_rows)
    best_dff_row = dff_threshold_df.loc[dff_threshold_df["f1"].idxmax()]

    print("Best ΔF/F threshold by F1:")
    display(best_dff_row.to_frame().T)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(dff_threshold_df["dff_threshold"], dff_threshold_df["precision"], label="Precision")
    ax.plot(dff_threshold_df["dff_threshold"], dff_threshold_df["recall"], label="Recall")
    ax.plot(dff_threshold_df["dff_threshold"], dff_threshold_df["f1"], label="F1")
    ax.axvline(0.25, linestyle="--", label="Example 0.25")
    ax.axvline(best_dff_row["dff_threshold"], linestyle=":", label=f"Best F1 {best_dff_row['dff_threshold']:.3f}")
    ax.set_xlabel(f"{dff_col} threshold")
    ax.set_ylabel("Score")
    ax.set_title("Traditional ΔF/F Threshold Scan")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No ΔF/F/amplitude column detected, so this section was skipped.")

## 12. Compare ΔF/F thresholds separately for uninjured and injured retina

This tests whether injured retina may need a different activity threshold than uninjured retina.


In [ ]:
if dff_col is not None:
    group_threshold_results = []

    for group_name, sub in test_df.groupby("retina_group"):
        candidate_thresholds = np.linspace(
            np.nanpercentile(sub[dff_col], 1),
            np.nanpercentile(sub[dff_col], 99),
            100
        )
        rows = []
        for thr in candidate_thresholds:
            pred = (sub[dff_col] >= thr).astype(int)
            rows.append({
                "retina_group": group_name,
                "dff_threshold": thr,
                "accuracy": accuracy_score(sub["y_true"], pred),
                "precision": precision_score(sub["y_true"], pred, zero_division=0),
                "recall": recall_score(sub["y_true"], pred, zero_division=0),
                "f1": f1_score(sub["y_true"], pred, zero_division=0),
                "predicted_active_fraction": pred.mean(),
            })
        group_df = pd.DataFrame(rows)
        best = group_df.loc[group_df["f1"].idxmax()]
        group_threshold_results.append(best)

    group_threshold_results = pd.DataFrame(group_threshold_results)
    display(group_threshold_results)
else:
    print("No ΔF/F/amplitude column detected, so group threshold comparison was skipped.")

## 13. Feature importance

This shows which waveform features were most useful for classifying active vs non-active.


In [ ]:
if hasattr(model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)

    display(importance_df.head(20))

    fig, ax = plt.subplots(figsize=(8, 6))
    top = importance_df.head(20).sort_values("importance")
    ax.barh(top["feature"], top["importance"])
    ax.set_title("Top Built-in Feature Importances")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.show()
else:
    print("This model does not expose built-in feature_importances_.")

In [ ]:
perm = permutation_importance(
    model, X_test, y_test,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="roc_auc",
    n_jobs=-1,
)

perm_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

display(perm_df.head(20))

fig, ax = plt.subplots(figsize=(8, 6))
top = perm_df.head(20).sort_values("importance_mean")
ax.barh(top["feature"], top["importance_mean"])
ax.set_title("Top Permutation Importances")
ax.set_xlabel("Mean decrease in ROC-AUC when shuffled")
plt.tight_layout()
plt.show()

## 14. Save model and outputs

This saves:

- trained model
- feature column list
- test predictions
- probability threshold scan
- ΔF/F threshold scan, if available


In [ ]:
model_path = OUTPUT_DIR / "xgboost_retina_activity_model.joblib"
features_path = OUTPUT_DIR / "model_feature_columns.joblib"
predictions_path = OUTPUT_DIR / "test_predictions.csv"
thresholds_path = OUTPUT_DIR / "probability_threshold_scan.csv"

joblib.dump(model, model_path)
joblib.dump(feature_cols, features_path)

results_df = df_model.loc[X_test.index].copy()
results_df["y_true"] = y_test.values
results_df["probability_active"] = y_prob
results_df["predicted_active_0p5"] = y_pred

results_df.to_csv(predictions_path, index=False)
threshold_df.to_csv(thresholds_path, index=False)

print("Saved model to:", model_path)
print("Saved feature columns to:", features_path)
print("Saved predictions to:", predictions_path)
print("Saved probability threshold scan to:", thresholds_path)

if dff_col is not None:
    dff_threshold_df.to_csv(OUTPUT_DIR / "dff_threshold_scan.csv", index=False)
    print("Saved ΔF/F threshold scan to:", OUTPUT_DIR / "dff_threshold_scan.csv")

## 15. Apply model to a new CSV

Use this template later to classify a new retina waveform CSV.


In [ ]:
# Example only. Uncomment and modify when you have a new file.
#
# new_path = DATA_DIR / "new_retina_waveforms.csv"
# new_df = pd.read_csv(new_path)
# new_df["retina_group"] = "unknown"
# new_df["retina_group_injured"] = 0
#
# new_df = add_trace_features(new_df, detect_trace_columns(new_df))
#
# loaded_model = joblib.load(OUTPUT_DIR / "xgboost_retina_activity_model.joblib")
# loaded_features = joblib.load(OUTPUT_DIR / "model_feature_columns.joblib")
#
# X_new = new_df[loaded_features].copy()
# X_new = X_new.replace([np.inf, -np.inf], np.nan)
# X_new = X_new.fillna(X_new.median(numeric_only=True))
#
# new_df["probability_active"] = loaded_model.predict_proba(X_new)[:, 1]
# new_df["predicted_active"] = (new_df["probability_active"] >= 0.5).astype(int)
#
# new_df.to_csv(OUTPUT_DIR / "new_retina_predictions.csv", index=False)
# display(new_df.head())

# Summary

This notebook implements a practical XGBoost workflow:

1. Load uninjured and injured retina waveform CSV files.
2. Use existing active/non-active labels, or create provisional labels from ΔF/F.
3. Optionally extract waveform features from raw trace columns.
4. Train XGBoost / Gradient Boosting.
5. Evaluate classification performance.
6. Choose a model probability threshold.
7. Compare ML prediction to a ΔF/F threshold such as 0.25.
8. Estimate separate thresholds for uninjured and injured retina.
9. Save the trained model and outputs.
